In [1]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import minimize

# вычисления для 7

In [2]:
def minus_log_L(teta):
    return -(np.exp(-196*teta) * teta**109 /2**22 * (1 - np.exp(-teta)*(1 + teta + teta**2/2))**4)

In [3]:
# начальные данные
teta0 = 1

# Оптимизация L
result = minimize(minus_log_L, teta0, method='Nelder-Mead')
teta = result.x[0]

print(f"Оценка (ОМПГ): {teta}")

Оценка (ОМПГ): 0.6083007812499996


In [4]:
p0 = np.exp(-teta)
p1 = teta * np.exp(-teta)
p2 = teta**2 * np.exp(-teta)/2
p3 = 1-(p0+p1+p2)

delta = (109 - 200*p0)**2/(200*p0) + (65 - 200*p1)**2/(200*p1) + (22 - 200*p2)**2/(200*p2) + (4 - 200*p3)**2/(200*p3)
print(delta)

0.32424443964938604


# 10b) вычисления для хи^2

In [5]:
n = 100
m_i = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])
x_i = np.arange(10)


In [6]:
def minus_log_L(params):
    teta1, teta2 = params
    ans = 0
    p = np.zeros(10)
    for i in range(10):
        x2 = ((i+1) - teta1)/teta2
        x1 = (i - teta1)/teta2
        p[i] = norm.cdf(x2) - norm.cdf(x1)
        if p[i] == 0 or p[i] < 0:
            return 10**(10)
        ans += m_i[i]*np.log(p[i])
    return -ans
    

In [7]:
# начальные данные (из ОММ)
mu0 = np.sum(m_i * x_i)/n
alfa_2 = np.sum(m_i * x_i**2)/n
sigma0 = np.sqrt(alfa_2 - mu0**2)

# Оптимизация L
result = minimize(minus_log_L, [mu0, sigma0], method='Nelder-Mead')
mu_est, sigma_est = result.x

print(f"Оценка mu (ОМПГ): {mu_est:.3f}")
print(f"Оценка sigma (ОМПГ): {sigma_est:.3f}")


Оценка mu (ОМПГ): 5.270
Оценка sigma (ОМПГ): 2.489


In [8]:
teta1 = 5.270
teta2 = 2.489

In [9]:
delta = 0
p_i = 0
for i in range(10):
    x2 = ((i+1) - teta1)/teta2
    x1 = (i - teta1)/teta2
    p_i = norm.cdf(x2) - norm.cdf(x1)

    delta += (m_i[i] - n*p_i)**2 / (n*p_i)

print(delta)

15.704007306951077


# 10b) Колмолоров

### оценка по омм

In [10]:
mean = np.sum(m_i * x_i)/n
alfa_2 = np.sum(m_i * x_i**2)/n

teta1 = mean
teta2 = np.sqrt(alfa_2 - mean**2)
print(teta1, teta2)

4.77 2.5054141374231933


In [11]:
def empirical_cdf(data, x_i):
    ans = np.zeros(np.size(x_i))
    for i in range(np.size(x_i)):
        ans[i] = np.sum(data <= x_i[i])/np.size(data)
    return ans

def empirical_cdf_left(data, x_i):
    ans = np.zeros(np.size(x_i))
    for i in range(np.size(x_i)):
        ans[i] = np.sum(data < x_i[i])/np.size(data)
    return ans

In [12]:
data = np.repeat(x_i, m_i)
delta_0 = np.sqrt(n) * max(max(np.abs(empirical_cdf(data, x_i) - norm.cdf(x_i, teta1, teta2))), 
                                    max(np.abs(empirical_cdf_left(data, x_i) - norm.cdf(x_i, teta1, teta2))))
print(delta_0)

1.013371112422481


In [13]:
def teta_est(x_n, n):
    teta1_est = np.mean(x_n)
    alfa_2 = np.sum(x_n**2)/n
    teta2_est = np.sqrt(alfa_2 - teta1_est**2)
    return teta1_est, teta2_est

def bootstrap(mu, sigma, n, N):
    delta = np.zeros(N)
    for i in range (0, N):
        sub_x = np.random.normal(mu, sigma, n)
        mu_est, sigma_est = teta_est(sub_x, n)

        x_vals = np.sort(np.unique(sub_x))
        delta[i] = np.sqrt(n) * max(max(np.abs(empirical_cdf(sub_x, x_vals) - norm.cdf(x_vals, mu_est, sigma_est))), 
                                    max(np.abs(empirical_cdf_left(sub_x, x_vals) - norm.cdf(x_vals, mu_est, sigma_est))))
    return delta

N = 100000
delta = bootstrap(teta1, teta2, n, N)

p_value = np.sum(delta >= delta_0)/N
print(p_value)

0.01331
